# 7.4 ??? ??? ???? ???? ?? ?? I

- [??? ??? ???? ???? ??](https://www.kaggle.com/c/cat-in-the-dat/)
- [??? ?? ?? ??](https://www.kaggle.com/dkomyagin/cat-in-the-dat-0-80285-private-lb-solution)

In [ ]:
import pandas as pd
# ??? ??
data_path = '/kaggle/input/cat-in-the-dat/'

train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

## 7.4.1 ?? ????? I : ?? ?? ???

### ??? ???

In [ ]:
# ?? ???? ??? ??? ???
all_data = pd.concat([train, test])
all_data = all_data.drop('target', axis=1) # ??? ??

### ?? ?? ???

In [ ]:
all_data['bin_3'] = all_data['bin_3'].map({'F':0, 'T':1})
all_data['bin_4'] = all_data['bin_4'].map({'N':0, 'Y':1})

### ??? ?? ???

In [ ]:
ord1dict = {'Novice':0, 'Contributor':1, 
            'Expert':2, 'Master':3, 'Grandmaster':4}
ord2dict = {'Freezing':0, 'Cold':1, 'Warm':2, 
            'Hot':3, 'Boiling Hot':4, 'Lava Hot':5}

all_data['ord_1'] = all_data['ord_1'].map(ord1dict)
all_data['ord_2'] = all_data['ord_2'].map(ord2dict)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ord_345 = ['ord_3', 'ord_4', 'ord_5']

ord_encoder = OrdinalEncoder() # OrdinalEncoder ?? ??
# ordinal ??? ??
all_data[ord_345] = ord_encoder.fit_transform(all_data[ord_345])

# ??? ??? ?? ??
for feature, categories in zip(ord_345, ord_encoder.categories_):
    print(feature)
    print(categories)

### ??? ?? ???

In [ ]:
nom_features = ['nom_' + str(i) for i in range(10)] # ??? ??

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder() # OneHotEncoder ?? ??
# ?-? ??? ??
encoded_nom_matrix = onehot_encoder.fit_transform(all_data[nom_features])

encoded_nom_matrix

In [ ]:
all_data = all_data.drop(nom_features, axis=1) # ?? ??? ?? ??

### ?? ?? ???

In [ ]:
date_features  = ['day', 'month'] # ?? ??

# ?-? ??? ??
encoded_date_matrix = onehot_encoder.fit_transform(all_data[date_features])

all_data = all_data.drop(date_features, axis=1) # ?? ?? ?? ??

encoded_date_matrix

## 7.4.2 ?? ????? II : ?? ????

### ??? ?? ????

In [ ]:
from sklearn.preprocessing import MinMaxScaler

ord_features = ['ord_' + str(i) for i in range(6)] # ??? ??
# min-max ???
all_data[ord_features] = MinMaxScaler().fit_transform(all_data[ord_features])

### ??? ? ????? ?? ???

In [ ]:
from scipy import sparse

# ??? ? ????? ?? ???
all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data),
                               encoded_nom_matrix,
                               encoded_date_matrix],
                              format='csr')

In [ ]:
all_data_sprs

### ??? ???

In [ ]:
num_train = len(train) # ?? ??? ??

# ?? ???? ??? ??? ???
X_train = all_data_sprs[:num_train] # 0 ~ num_train - 1?
X_test = all_data_sprs[num_train:] # num_train ~ ??? ?

y = train['target']

In [ ]:
from sklearn.model_selection import train_test_split

# ?? ???, ?? ??? ??
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y,
                                                      test_size=0.1,
                                                      stratify=y,
                                                      random_state=10)

## 7.4.3 ??? ???? ???

In [ ]:
%%time

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# ???? ?? ?? ??
logistic_model = LogisticRegression()

# ??????? ? ??
lr_params = {'C':[0.1, 0.125, 0.2], 'max_iter':[800, 900, 1000], 
             'solver':['liblinear'], 'random_state':[42]}

# ????? ?? ??
gridsearch_logistic_model = GridSearchCV(estimator=logistic_model,
                                         param_grid=lr_params,
                                         scoring='roc_auc', # ????
                                         cv=5)
# ????? ??
gridsearch_logistic_model.fit(X_train, y_train)

print('?? ???????:', gridsearch_logistic_model.best_params_)

## 7.4.4 ?? ?? ??

In [ ]:
y_valid_preds = gridsearch_logistic_model.predict_proba(X_valid)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score # ROC AUC ?? ?? ??

# ?? ??? ROC AUC
roc_auc = roc_auc_score(y_valid, y_valid_preds)

print(f'?? ??? ROC AUC : {roc_auc:.4f}')

## 7.4.5 ?? ? ?? ??

In [ ]:
# ??? 1? ?? ??
y_preds = gridsearch_logistic_model.best_estimator_.predict_proba(X_test)[:,1]

# ?? ?? ??
submission['target'] = y_preds
submission.to_csv('submission.csv')